<a href="https://colab.research.google.com/github/sadar-naman/AI-Core-Reforged/blob/main/02-Pandas-Mastery/02_ENTERPRISE_DATA_ENGINEERING_%26_ANALYTICSfl_PIPELINE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# AI_CORE_REFORGED: ENTERPRISE DATA ENGINEERING & ANALYTICS PIPELINE
# Dataset: Supermarket Sales (Kaggle Retail Data)
# Architecture: Vectorized Pandas, Zero-Loops, Defensive Processing
# ==============================================================================

# ------------------------------------------------------------------------------
# LIBRARIES INGESTION
# ------------------------------------------------------------------------------
import pandas as pd
import numpy as np
import io
from google.colab import files

# ==============================================================================
# PHASE 1: DATA INGESTION & STRUCTURAL INSPECTION
# ==============================================================================
print("अपनी डाउनलोड की हुई CSV फाइल को सिलेक्ट करें:")
# यह लाइन कोलाब में सीधे एक 'Choose Files' का बटन बना देगी
uploaded = files.upload()

# जो भी फाइल अपलोड होगी, उसका नाम यह कोड खुद ढूंढ लेगा (नाम गलत होने का झंझट खत्म)
file_name = list(uploaded.keys())[0]

# डेटा लोड करना
sales_raw = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\n✅ असली कैगल डेटा सफलतापूर्वक लोड हो गया है!")
print(f"फाइल का नाम: {file_name}")
print(f"डेटा का आकार (Rows, Columns): {sales_raw.shape}\n")
print(sales_raw.head())
print(sales_raw.info())


# ==============================================================================
# PHASE 2: DATA CLEANING & CUSTOM VALIDATION CHECKS
# ==============================================================================

# [कठिन लॉजिक - 1: Date Parsing with Coerce]
# 'errors=coerce' का मतलब है कि अगर तारीख के फॉर्मेट में कोई कबाड़ या अमान्य एंट्री होगी,
# तो कोड क्रैश होने के बजाय उसे चुपचाप NaT (Not a Time/Null) बना देगा।
sales_raw['Date'] = pd.to_datetime(sales_raw.Date, errors='coerce')

# [कठिन लॉजिक - 2: Idempotent Time Parsing Guard]
# idempotency सुनिश्चित करने के लिए (ताकि सेल को बार-बार रन करने पर एरर न आए):
# यह चेक करता है कि क्या 'Time' कॉलम खाली नहीं है और उसका पहला एलिमेंट String (Text) है।
# केवल तभी यह रूपांतरण (Conversion) चलाएगा।
if sales_raw['Time'].notna().any() and isinstance(sales_raw['Time'].dropna().iloc[0], str):
    # 'format=mixed' बिना क्रैश हुए विभिन्न टाइम फॉरमेट्स (HH:MM या HH:MM:SS) को ऑटो-डिटेक्ट करता है।
    # '.dt.time' तारीख वाले हिस्से को हटाकर सिर्फ शुद्ध समय (datetime.time ऑब्जेक्ट) छोड़ता है।
    sales_raw['Time'] = pd.to_datetime(sales_raw.Time, format='mixed').dt.time

# Correct the typo 'whare' to 'where' and the column selection
# Assuming the user wants to select rows where 'Quantity' or 'Unit price' is <= 0
# and display those specific columns.
# Using .loc for filtering rows based on condition and selecting specific col

# [कठिन लॉजिक - 3: Vectorized Data Integrity Inspection]
# यह बिना किसी 'for loop' के सीधे पूरे डेटाफ्रेम के सभी न्यूमेरिक कॉलम्स को स्कैन करता है
# और चेक करता है कि क्या कहीं भी कोई वैल्यू 0 या निगेटिव (<= 0) है।
print((sales_raw.select_dtypes(include='number') <= 0).any())

# [कठिन लॉजिक - 4: Floating-Point Math Verification]
# कंप्यूटर में floating-point arithmetic की वजह से (उदा. 0.05 * 10 = 0.50000000001) सीधा '==' फेल हो सकता है।
# इसलिए 'np.isclose()' का उपयोग किया जाता है।
# '~' (bitwise NOT operator) लगाकर हम उन रोज़ (Rows) को ढूंढ रहे हैं जहाँ 5% टैक्स का हिसाब गलत है।
print(sales_raw[~np.isclose(sales_raw['Unit price']*sales_raw['Quantity']*0.05, sales_raw['Tax 5%'])])


# ==============================================================================
# PHASE 3: FEATURE ENGINEERING (ADVANCED ATRIBUTES CREATION)
# ==============================================================================

# 1. High-Value Vectorized Boolean Mask
# ₹500 से ज़्यादा के कुल ट्रांजेक्शन को True/False का बुलियन फ्लैग देता है।
sales_raw['Is_High_Value'] = sales_raw['Total'] > 500

# 2. Time-Series Date Extraction
sales_raw['Month_Name'] = sales_raw['Date'].dt.month_name()

# 3. Peak Hour Extraction
# पंडास में Pure Time 'object' पर सीधे .dt.hour काम नहीं करता,
# इसलिए पहले उसे स्ट्रिंग फॉरमेट में पार्स करके उसका 24-घंटे वाला घंटा (Hour) निकाला गया है।
sales_raw['Peak_Hour'] = pd.to_datetime(sales_raw['Time'], format='%H:%M:%S').dt.hour


# ==============================================================================
# PHASE 4: EXECUTIVE BUSINESS INTELLIGENCE & AGGREGATIONS
# ==============================================================================

# [कठिन लॉजिक - 5: Multi-Level GroupBy Aggregation]
# जैसे: दुकान की लोकेशन (Branch) और जेंडर (Gender) दोनों के हिसाब से सेल्स का हिसाब।
print(sales_raw.groupby(['Branch', 'Gender'])[['Quantity', 'Total']].sum())

# [कठिन लॉजिक - 6: Pivot Table Matrix]
# Product line के आधार पर अलग-अलग Payment Mode का औसतन (Mean) रेवेन्यू दिखाने वाली समरी।
table = pd.pivot_table(
    sales_raw,
    values='Total',          # जिस कॉलम की वैल्यूज का हिसाब लगाना है
    index='Product line',    # जो रोज़ (Rows) में दिखेगा
    columns='Payment',       # जो कॉलम्स (Columns) में दिखेगा
    aggfunc='mean'           # कौन-सा ऑपरेशन करना है (बाय-डिफ़ॉल्ट 'mean' होता है)
)

print(table)

# [कठिन लॉजिक - 7: Peak Shopping Hour Identification]
# 1. हर घंटे की टोटल बिक्री का जोड़ (Revenue per Hour) निकाला।
hourly_revenue = sales_raw.groupby('Peak_Hour')['Total'].sum()

# 2. '.idxmax()' पूरे सीरीज़ में घूमकर उस इंडेक्स (Hour Value) को निकालता है जहाँ रेवेन्यू मैक्सिमम (Peak) था।
peak_hour_val = hourly_revenue.idxmax()

print(f"पीक शॉपिंग आवर: {peak_hour_val}")

print(sales_raw.head())


# ==============================================================================
# PHASE 5: PRODUCTION EXPORT
# ==============================================================================
# cleaned_df को अपनी CSV फाइल में सेव करने के लिए
# index=False से ऑटो-जनरेटेड 0, 1, 2... इंडेक्स कॉलम फाइल में सेव नहीं होता।
sales_raw.to_csv('cleaned_supermarket_sales.csv', index=False)